# Backward Model Hyperparameter Tuning (Wrought only)

Tunes GMM (n_components, covariance_type) using BIC on the **wrought** composition space. Saves best params to `backward.wrought.GMM` in `hyperparams_config.json`.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')

try:
    from utils import load_hyperparams, save_hyperparams, get_default_hyperparams
except ImportError:
    import sys
    sys.path.insert(0, os.getcwd())
    from utils import load_hyperparams, save_hyperparams, get_default_hyperparams

INPUT_COLS = ['Al', 'Si', 'Fe', 'Cu', 'Mn', 'Mg', 'Cr', 'Ni', 'Zn', 'Ga', 'V', 'Ti']

In [2]:
# Load wrought composition data (12 element columns only)
WROUGHT_PATH = 'wrought_alloys_final.csv'

def load_wrought_compositions(path):
    if not os.path.exists(path):
        return None
    if path.endswith('.xlsx') or path.endswith('.xls'):
        df = pd.read_excel(path)
    else:
        try:
            with open(path, 'rb') as f:
                if f.read(2) == b'PK':
                    df = pd.read_excel(path)
                else:
                    try:
                        df = pd.read_csv(path, encoding='utf-8')
                    except UnicodeDecodeError:
                        df = pd.read_csv(path, encoding='latin-1')
        except Exception:
            df = pd.read_excel(path)
    for c in INPUT_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    df[INPUT_COLS] = df[INPUT_COLS].fillna(0.0)
    return df[INPUT_COLS].dropna()

X_wrought = load_wrought_compositions(WROUGHT_PATH)
if X_wrought is not None:
    print(f'Wrought composition data: {X_wrought.shape}')
else:
    print('Wrought data not found.')

Wrought composition data: (868, 12)


In [3]:
# Tune GMM per dataset: n_components and covariance_type using BIC
n_components_range = [6, 8, 10, 12, 15, 20]
covariance_types = ['full', 'tied']

def tune_gmm_bic(X, dataset_name):
    if X is None or len(X) < 20:
        return None
    best_bic = np.inf
    best_params = None
    results = []
    for n in n_components_range:
        for cov in covariance_types:
            try:
                gmm = GaussianMixture(n_components=n, covariance_type=cov, random_state=42)
                gmm.fit(X)
                bic = gmm.bic(X)
                results.append({'n_components': n, 'covariance_type': cov, 'bic': bic})
                if bic < best_bic:
                    best_bic = bic
                    best_params = {'n_components': n, 'covariance_type': cov, 'random_state': 42}
            except Exception as e:
                print(f'  Skip n={n} cov={cov}: {e}')
    print(f'{dataset_name} GMM BIC (top 3):')
    for r in sorted(results, key=lambda x: x['bic'])[:3]:
        print(f"  n={r['n_components']}, cov={r['covariance_type']}: BIC={r['bic']:.0f}")
    print(f'  Best: {best_params}\n')
    return best_params

best_wrought_gmm = tune_gmm_bic(X_wrought, 'Wrought')

Wrought GMM BIC (top 3):
  n=20, cov=full: BIC=-56134
  n=15, cov=full: BIC=-47276
  n=12, cov=full: BIC=-42195
  Best: {'n_components': 20, 'covariance_type': 'full', 'random_state': 42}



In [4]:
# Save backward GMM params (wrought only; merge into config)
if best_wrought_gmm is not None:
    save_hyperparams({'backward': {'wrought': {'GMM': best_wrought_gmm}}})
    print('Saved backward.wrought.GMM to hyperparams_config.json')
else:
    default_gmm = get_default_hyperparams('GMM')
    save_hyperparams({'backward': {'wrought': {'GMM': default_gmm}}})
    print('No wrought data; saved default backward.wrought.GMM')

# Forward per-target params are used by 05_backward_wrought and synthetic generator 06_wrought
by_target_w = load_hyperparams('wrought') and load_hyperparams('wrought').get('by_target')
if by_target_w:
    print('Forward wrought.by_target available for backward/synthetic pipelines.')
else:
    print('Run 01_hyperparameter_tuning_forward.ipynb first for wrought.by_target.')

Saved backward.wrought.GMM to hyperparams_config.json
Forward wrought.by_target available for backward/synthetic pipelines.
